In [2]:
# %% [markdown]
# # 07 — Polymodel (LNLM + AVA)
#
# Nonlinear additive baseline: one univariate LNLM per feature,
# aggregated via Added Value Averaging (Barrau & Douady, 2022).
#
# Always fits on continuous target (minret_5d_pct — raw five-day block
# minimum in percentage points).
# Binary metrics derived by ranking predictions against y_binary labels.
# No sigmoid, no BCE — pure regression.
#
# Target notes:
#   Continuous target is minret_5d_pct, in percentage points. MSE is therefore
#   in percent² and directly comparable to the volatility baseline.
#   derive_binary_from_continuous receives y_binary (0/1 labels) as first
#   argument — thresholding predictions is meaningless; ranking is the point.
#   Derived AUC uses -y_pred as ranking score against y_binary.
#
# Reads the Stage 5 splits directly rather than through data_utils, since the
# continuous target changed from an expanding z-score to raw percentage points.
#
# LNLM implementation matches existing polymodel code:
#   - Target centered (y - mean(y)) before fitting
#   - 4 Hermite polynomials (He_1 through He_4), no intercept
#   - 101 μ values (0.00 to 1.00 in 0.01 steps)
#   - Stratified 10-fold CV for μ selection
#   - Post-fit cleanup for explosive norms and inf/NaN
#
# Four aggregation variants (same fitted LNLMs, different weighting):
#   1. Equal average
#   2. RMSE-weighted (believability only)
#   3. AVA (believability × originality)
#   4. AVA + uncertainty scaling
#
# Backtest uses the AVA variant — the theoretically strongest aggregation.
# run_full_backtest produces: signal diagnostics, simple timing, timing
# cost sweep, asymmetric risk-scaled, risk-scaled cost sweep, side-by-side.
# All other variants are evaluated and saved; their metrics appear in the
# summary tables above the backtest section.
#
# 4 splits × 2 feature sets × 4 variants = 32 prediction sets
# Only 8 LNLM fitting runs (4 splits × 2 feature sets).
#
# Expected runtime: < 15 minutes on local CPU.

# %%
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import time
from pathlib import Path

from scipy.stats import gaussian_kde
from sklearn.model_selection import KFold

from evaluation import (
    compute_continuous_metrics,
    derive_binary_from_continuous,
    save_predictions,
    load_predictions,
    run_full_backtest,
)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = (Path("../../..") / "Data" / "Data_Collection" / "Final"
               / "Stage_5_Model_Ready" / "04_splits")
RESULTS_DIR = Path("../../..") / "Data" / "Results" / "Dense_vs_Sparse_KAN" / "Polymodel"

FEATURE_SETS = ["agg_full_moments", "agg_means"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

# ── Meta columns per dataset ──
# Everything not listed here is a model feature, including the five binary
# regime indicators, which pass through un-z-scored.
META = {
    "agg_means":        ["date", "target_daily_return", "minret_5d_pct", "y_binary"],
    "agg_full_moments": ["date", "target_daily_return", "minret_5d_pct", "y_binary"],
    "panel":            ["permno", "date", "dlyret", "dlycap",
                         "minret_5d_pct", "y_binary"],
}

# LNLM settings (matching existing polymodel code)
N_HERMITE       = 4                       # He_1 through He_4
N_FOLDS         = 10                      # CV folds for μ selection
MU_GRID         = np.linspace(0, 1, 101)  # 0.00, 0.01, ..., 1.00
NORM_THRESHOLD  = 50                      # exclude features with explosive β_nonlin
FILTER_BY_NAIVE = True

# ── Believability normalisation ──
# B_p = [ln(RMSE_p)]² is symmetric about RMSE = 1: it falls as RMSE rises
# toward 1, then rises again beyond it, so it only ranks models correctly while
# RMSE < 1. With the old z-scored target (sd ≈ 1) every surviving feature sat
# in that region. With minret_5d_pct (sd ≈ 1.23), any feature with R² below
# roughly 0.34 has RMSE > 1, which is essentially all of them, and a WORSE
# model would receive a HIGHER weight.
#
# Dividing by the naive RMSE before taking the log restores monotonicity: every
# feature passing the filter has RMSE_p < naive_RMSE by construction, so the
# ratio is always below 1. It also makes believability scale-free, which is
# arguably what it should have been. Set False to recover the original formula.
NORMALISE_BELIEVABILITY = True

# NOTE on NORM_THRESHOLD: coefficients scale with the target, so the 1.23×
# larger target means this threshold now bites about 81% as readily as before.
# The constant was arbitrary to begin with, so it is left unchanged.
#
# NOTE on originality: [ln(density)]² from a per-day KDE is also scale-
# sensitive. Predictions scale by roughly 1.23, so densities fall by about the
# same factor and ln(density) shifts by around −0.21. The fix is not obvious
# and the effect is second-order, so this is flagged rather than changed.

# Numerical floors
RMSE_FLOOR    = 1e-10
DENSITY_FLOOR = 1e-100


# ═══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════

def load_split(split_name: str, dataset: str, splits_dir: Path = SPLITS_DIR) -> dict:
    """
    Load one split of one dataset from the Stage 5 output.

    The continuous target is minret_5d_pct, the raw five-day block minimum in
    percentage points, not the expanding z-score the previous pipeline used.
    Nothing is standardised here: the feature matrix arrives already clipped to
    ±5 and cast to float32 by 05_splits.ipynb.

    target_daily_return is pre-shifted, so row t holds the return earned on day
    t+1. A signal formed at t and applied to this column is therefore correctly
    aligned for the backtest with no further shifting.
    """
    out, meta = {}, META[dataset]

    for part in ("train", "val", "test"):
        df = pd.read_parquet(splits_dir / split_name / f"{dataset}_{part}.parquet")
        feats = [c for c in df.columns if c not in meta]

        out[f"X_{part}"]      = df[feats].to_numpy(dtype=np.float32)
        out[f"y_{part}"]      = df["y_binary"].to_numpy(dtype=np.float32)
        out[f"minret_{part}"] = df["minret_5d_pct"].to_numpy(dtype=np.float32)
        out[f"dates_{part}"]  = df["date"].reset_index(drop=True)

        rcol = "dlyret" if dataset == "panel" else "target_daily_return"
        out[f"returns_{part}"] = df[rcol].to_numpy(dtype=np.float64)

        if dataset == "panel":
            out[f"permno_{part}"] = df["permno"].to_numpy()

    out["feature_cols"] = feats
    out["n_features"]   = len(feats)
    return out


# ═══════════════════════════════════════════════════════════════════════════════
# HERMITE BASIS
# ═══════════════════════════════════════════════════════════════════════════════

def build_hermite_matrix(x):
    """
    Probabilist's Hermite polynomials He_1 through He_4, no intercept.
    Matches existing polymodel code exactly.
    """
    x2 = x * x
    return np.column_stack([
        x,
        x2 - 1,
        x2 * x - 3 * x,
        x2 * x2 - 6 * x2 + 3,
    ])


# ═══════════════════════════════════════════════════════════════════════════════
# STRATIFIED K-FOLD
# ═══════════════════════════════════════════════════════════════════════════════

def stratified_kfold(y, k=10, seed=42):
    """
    Sort by y, assign fold labels cyclically: 0,1,...,k-1,0,1,...
    Ensures each fold has a similar distribution of y values.
    """
    n = len(y)
    sorted_indices = np.argsort(y)
    folds = np.zeros(n, dtype=int)
    for i, idx in enumerate(sorted_indices):
        folds[idx] = i % k
    return folds


# ═══════════════════════════════════════════════════════════════════════════════
# LNLM FITTING
# ═══════════════════════════════════════════════════════════════════════════════

def fit_single_factor(x, y_centered, fold_assignments, k=10):
    """
    Fit one LNLM on centered target via k-fold CV over 101 μ values.
    Model: f(x) = (1-μ) * β_lin * x + μ * H @ β_nonlin
    No intercept — target already centered.
    Returns (mu_star, beta_lin, beta_nonlin).
    """
    cv_errors = np.zeros(len(MU_GRID))

    for fold in range(k):
        train_mask = fold_assignments != fold
        val_mask   = fold_assignments == fold
        if val_mask.sum() == 0:
            continue

        x_tr, x_va = x[train_mask], x[val_mask]
        y_tr, y_va = y_centered[train_mask], y_centered[val_mask]

        denom = np.dot(x_tr, x_tr)
        bl    = np.dot(x_tr, y_tr) / denom if denom > 1e-20 else 0.0

        H_tr = build_hermite_matrix(x_tr)
        H_va = build_hermite_matrix(x_va)
        bn, _, _, _ = np.linalg.lstsq(H_tr, y_tr, rcond=None)

        lin_pred = bl * x_va
        nl_pred  = H_va @ bn

        for i, mu in enumerate(MU_GRID):
            combined      = (1 - mu) * lin_pred + mu * nl_pred
            cv_errors[i] += np.sum((y_va - combined) ** 2)

    best_idx = np.argmin(cv_errors)
    mu_star  = MU_GRID[best_idx]

    denom      = np.dot(x, x)
    beta_lin   = np.dot(x, y_centered) / denom if denom > 1e-20 else 0.0
    H          = build_hermite_matrix(x)
    beta_nonlin, _, _, _ = np.linalg.lstsq(H, y_centered, rcond=None)

    return mu_star, beta_lin, beta_nonlin


def fit_all_lnlms(X_train, y_train, feature_cols):
    """
    Fit one LNLM per feature on the training set.
    Target is centered once. Fold assignments shared across features.
    Returns list of model dicts + y_train_mean.
    """
    n_samples, n_features = X_train.shape
    y_mean     = np.mean(y_train)
    y_centered = y_train - y_mean

    fold_assignments = stratified_kfold(y_centered, k=N_FOLDS, seed=42)

    models     = []
    n_screened = 0

    for p in range(n_features):
        x = X_train[:, p]

        if np.std(x) < 1e-10 or np.sum(np.isfinite(x)) < 100:
            models.append({
                "beta_lin": 0.0, "beta_nonlin": np.zeros(N_HERMITE),
                "mu": 0.0, "valid": False,
            })
            n_screened += 1
            continue

        try:
            ms, bl, bn = fit_single_factor(x, y_centered, fold_assignments, k=N_FOLDS)
            models.append({
                "beta_lin": bl, "beta_nonlin": bn, "mu": ms, "valid": True,
            })
        except Exception:
            models.append({
                "beta_lin": 0.0, "beta_nonlin": np.zeros(N_HERMITE),
                "mu": 0.0, "valid": False,
            })
            n_screened += 1

        if (p + 1) % 500 == 0 or p == 0:
            print(f"    Fitted {p + 1}/{n_features} "
                  f"(μ={models[-1]['mu']:.2f} for {feature_cols[p][:35]})")

    print(f"    Screened out: {n_screened}/{n_features}")
    return models, y_mean


def lnlm_predict(x, model, y_mean):
    """y = y_mean + (1-μ) * β_lin * x + μ * H @ β_nonlin"""
    mu  = model["mu"]
    lin = model["beta_lin"] * x
    H   = build_hermite_matrix(x)
    nl  = H @ model["beta_nonlin"]
    return y_mean + (1 - mu) * lin + mu * nl


def predict_all(X, models, y_mean):
    """Generate predictions for all features. Shape (n_days, n_features)."""
    n_days, n_features = X.shape
    preds = np.zeros((n_days, n_features))
    for p in range(n_features):
        preds[:, p] = lnlm_predict(X[:, p], models[p], y_mean)
    return preds


# ═══════════════════════════════════════════════════════════════════════════════
# POST-FIT CLEANUP
# ═══════════════════════════════════════════════════════════════════════════════

def cleanup_models(models, feature_cols):
    """Remove invalid, explosive, and inf/NaN fits. Returns boolean mask."""
    n_features = len(models)
    keep       = np.ones(n_features, dtype=bool)

    n_invalid = n_explosive = n_inf = 0

    for p in range(n_features):
        m = models[p]
        if not m["valid"]:
            keep[p] = False; n_invalid += 1; continue
        bn_norm = np.linalg.norm(m["beta_nonlin"])
        if bn_norm > NORM_THRESHOLD:
            keep[p] = False; n_explosive += 1; continue
        if not np.all(np.isfinite(m["beta_lin"])):
            keep[p] = False; n_inf += 1; continue
        if not np.all(np.isfinite(m["beta_nonlin"])):
            keep[p] = False; n_inf += 1; continue

    print(f"    Cleanup: {n_invalid} invalid, {n_explosive} explosive, "
          f"{n_inf} inf/NaN → {keep.sum()}/{n_features} kept")
    return keep


# ═══════════════════════════════════════════════════════════════════════════════
# FILTERING
# ═══════════════════════════════════════════════════════════════════════════════

def filter_by_naive_rmse(val_preds, y_val, y_train_mean, clean_mask):
    """Keep features whose validation RMSE beats predicting the training mean."""
    n_features = val_preds.shape[1]
    naive_rmse = np.sqrt(np.mean((y_val - y_train_mean) ** 2))

    feature_rmse = np.full(n_features, np.nan)
    for p in range(n_features):
        if clean_mask[p]:
            feature_rmse[p] = np.sqrt(np.mean((y_val - val_preds[:, p]) ** 2))

    passes_filter = (feature_rmse < naive_rmse) & clean_mask
    print(f"    RMSE filter: {passes_filter.sum()}/{clean_mask.sum()} clean features pass "
          f"(naive RMSE = {naive_rmse:.4f})")
    return passes_filter, feature_rmse, naive_rmse


# ═══════════════════════════════════════════════════════════════════════════════
# BELIEVABILITY AND ORIGINALITY
# ═══════════════════════════════════════════════════════════════════════════════

def compute_believability(feature_rmse_filtered, naive_rmse=None):
    """
    B_p = [ln(RMSE_p)]², or [ln(RMSE_p / naive_RMSE)]² when normalised.

    See the NORMALISE_BELIEVABILITY note in the config block. In short,
    [ln(x)]² only ranks models correctly while x < 1, and with the percentage-
    point target most surviving features have RMSE above 1. Dividing by the
    naive RMSE guarantees the ratio is below 1 for every feature that passed
    the filter, restoring monotonicity and making the statistic scale-free.
    """
    rmse_floored = np.maximum(feature_rmse_filtered, RMSE_FLOOR)
    if NORMALISE_BELIEVABILITY and naive_rmse is not None:
        rmse_floored = rmse_floored / max(naive_rmse, RMSE_FLOOR)
    return np.log(rmse_floored) ** 2


def compute_originality(val_preds_filtered):
    """Per-day KDE, Shannon self-information, averaged over val days."""
    n_days, n_filtered = val_preds_filtered.shape
    originality_daily  = np.zeros((n_days, n_filtered))

    for t in range(n_days):
        day_preds = val_preds_filtered[t, :]
        if np.std(day_preds) < 1e-12:
            continue
        kde       = gaussian_kde(day_preds)
        densities = np.maximum(kde(day_preds), DENSITY_FLOOR)
        originality_daily[t, :] = np.log(densities) ** 2

        if (t + 1) % 100 == 0:
            print(f"      Originality: day {t + 1}/{n_days}")

    return np.mean(originality_daily, axis=0)


# ═══════════════════════════════════════════════════════════════════════════════
# WEIGHTS AND AGGREGATION
# ═══════════════════════════════════════════════════════════════════════════════

def compute_ava_weights(B, O):
    av    = B * O
    total = av.sum()
    return av / total if total > 1e-20 else np.ones(len(av)) / len(av)


def compute_rmse_weights(B):
    total = B.sum()
    return B / total if total > 1e-20 else np.ones(len(B)) / len(B)


def aggregate_predictions(preds_filtered, w_rmse, w_ava, sigma_bar, y_mean):
    """Four aggregation variants."""
    results            = {}
    results["equal"]   = np.mean(preds_filtered, axis=1)
    results["rmse"]    = preds_filtered @ w_rmse
    results["ava"]     = preds_filtered @ w_ava
    sigma_t            = np.maximum(np.std(preds_filtered, axis=1), 1e-10)
    # Scale the DEVIATION from the training mean, not the level. Dividing the
    # uncentred prediction shifts the level by an amount driven purely by
    # cross-sectional disagreement among the LNLMs, which is unrelated to the
    # target. Harmless when the target was z-scored and centred near zero;
    # catastrophic now the mean sits near -1.2.
    results["ava_unc"] = y_mean + (results["ava"] - y_mean) / (sigma_t / sigma_bar)
    return results


# ═══════════════════════════════════════════════════════════════════════════════
# EVALUATE AND SAVE
# ═══════════════════════════════════════════════════════════════════════════════

def evaluate_and_save_variant(variant_name, y_pred_dict, data,
                              split_name, feature_set, results_dir):
    """
    Evaluate one aggregation variant on train/val/test and save.

    y_true and y_pred are both minret_5d_pct in percentage points, so MSE is in
    percent² and directly comparable to the volatility baseline.
    derive_binary_from_continuous receives data["y_{part}"] (binary 0/1 labels)
    as the first argument — the model predicts a magnitude, not a class, and
    the derived AUC measures ranking rather than any threshold rule.
    """
    model_name  = f"polymodel_{variant_name}_{feature_set}"
    all_metrics = {}

    for part, pred in y_pred_dict.items():
        y_true   = data[f"minret_{part}"]   # percentage points
        y_binary = data[f"y_{part}"]        # binary 0/1 labels
        dates    = data[f"dates_{part}"]
        returns  = data[f"returns_{part}"]

        metrics = compute_continuous_metrics(y_true, pred)

        # Pass y_binary explicitly — predictions are a magnitude, not a class
        derived = derive_binary_from_continuous(y_binary, pred)
        metrics.update(derived)

        metrics["y_true"] = y_true
        metrics["y_pred"] = pred

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type="continuous",
            part=part,
            dates=dates,
            returns=returns,
            metrics=metrics,
            hyperparameters={
                "variant":   variant_name,
                "n_hermite": N_HERMITE,
                "n_folds":   N_FOLDS,
                "n_mu":      len(MU_GRID),
                "believability_normalised": NORMALISE_BELIEVABILITY,
            } if part == "test" else None,
            results_dir=results_dir,
            # Pass binary labels so parquet contains y_true_binary column
            y_true_binary=y_binary,
        )
        all_metrics[part] = metrics

    return all_metrics


# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════════

def run_polymodel_experiment(split_name, feature_set):
    """One split × one feature set. Fits LNLMs once, produces all 4 variants."""
    print(f"\n{'═'*60}")
    print(f"  {feature_set} / {split_name}")
    print(f"{'═'*60}")

    data     = load_split(split_name, feature_set, SPLITS_DIR)
    X_train  = data["X_train"]
    X_val    = data["X_val"]
    X_test   = data["X_test"]
    y_train  = data["minret_train"]   # percentage points
    y_val    = data["minret_val"]     # percentage points

    feature_cols = data["feature_cols"]
    n_features   = data["n_features"]

    print(f"  Features: {n_features}")
    print(f"  Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, "
          f"Test: {X_test.shape[0]}")
    print(f"  y_train range: [{y_train.min():.2f}, {y_train.max():.2f}] "
          f"(percentage points), sd={y_train.std():.3f}")

    # ── Step 1: Fit LNLMs ──
    print(f"\n  Fitting {n_features} LNLMs (101 μ values, 10-fold CV)...")
    t0 = time.time()
    models, y_mean = fit_all_lnlms(X_train, y_train, feature_cols)
    print(f"  Fitting complete in {time.time() - t0:.1f}s")

    mus = [m["mu"] for m in models if m["valid"]]
    print(f"  μ distribution: mean={np.mean(mus):.2f}, median={np.median(mus):.2f}, "
          f"linear(μ=0): {sum(1 for m in mus if m == 0)}, "
          f"nonlinear(μ=1): {sum(1 for m in mus if m >= 0.99)}")

    # ── Step 2: Cleanup ──
    print(f"\n  Post-fit cleanup...")
    clean_mask = cleanup_models(models, feature_cols)

    # ── Step 3: Predictions ──
    print(f"\n  Generating predictions...")
    train_preds = predict_all(X_train, models, y_mean)
    val_preds   = predict_all(X_val,   models, y_mean)
    test_preds  = predict_all(X_test,  models, y_mean)

    # ── Step 4: RMSE filter ──
    if FILTER_BY_NAIVE:
        final_mask, feature_rmse, naive_rmse = filter_by_naive_rmse(
            val_preds, y_val, y_mean, clean_mask
        )
    else:
        final_mask   = clean_mask.copy()
        feature_rmse = np.array([
            np.sqrt(np.mean((y_val - val_preds[:, p]) ** 2))
            if clean_mask[p] else np.nan
            for p in range(n_features)
        ])
        naive_rmse = np.sqrt(np.mean((y_val - y_mean) ** 2))

    if final_mask.sum() < 5:
        print(f"    WARNING: Only {final_mask.sum()} features passed. "
              f"Falling back to clean_mask ({clean_mask.sum()} features).")
        final_mask = clean_mask.copy()

    n_filtered      = final_mask.sum()
    train_preds_f   = train_preds[:, final_mask]
    val_preds_f     = val_preds[:,   final_mask]
    test_preds_f    = test_preds[:,   final_mask]
    feature_rmse_f  = feature_rmse[final_mask]
    feature_cols_f  = [feature_cols[i] for i in range(n_features) if final_mask[i]]

    # ── Step 5: Believability ──
    print(f"\n  Believability ({n_filtered} features)"
          f"{', normalised by naive RMSE' if NORMALISE_BELIEVABILITY else ''}...")
    B = compute_believability(feature_rmse_f, naive_rmse)

    # ── Step 6: Originality ──
    print(f"  Originality (KDE per day, {val_preds_f.shape[0]} val days)...")
    t0 = time.time()
    O  = compute_originality(val_preds_f)
    print(f"  Originality complete in {time.time() - t0:.1f}s")

    # ── Step 7: Weights ──
    w_rmse = compute_rmse_weights(B)
    w_ava  = compute_ava_weights(B, O)

    print(f"\n  Weights:")
    print(f"    RMSE: top10 share={np.sort(w_rmse)[-10:].sum():.3f}")
    print(f"    AVA:  top10 share={np.sort(w_ava)[-10:].sum():.3f}")

    # ── Step 8: Aggregate ──
    sigma_val = np.std(val_preds_f, axis=1)
    sigma_bar = max(np.mean(sigma_val), 1e-10)

    agg_train = aggregate_predictions(train_preds_f, w_rmse, w_ava, sigma_bar, y_mean)
    agg_val   = aggregate_predictions(val_preds_f,   w_rmse, w_ava, sigma_bar, y_mean)
    agg_test  = aggregate_predictions(test_preds_f,  w_rmse, w_ava, sigma_bar, y_mean)

    # ── Step 9: Evaluate and save ──
    all_variant_results = {}
    for variant in ["equal", "rmse", "ava", "ava_unc"]:
        y_pred_dict = {
            "train": agg_train[variant],
            "val":   agg_val[variant],
            "test":  agg_test[variant],
        }
        metrics = evaluate_and_save_variant(
            variant, y_pred_dict, data, split_name, feature_set, RESULTS_DIR,
        )
        all_variant_results[variant] = metrics

        test_m = metrics["test"]
        print(f"  {variant:>8}: R²={test_m['r2']:.4f}  "
              f"MSE={test_m['mse']:.4f}  "
              f"Derived AUC={test_m.get('derived_auc', 0):.4f}  "
              f"Pred std={test_m.get('pred_std', 0):.4f}")

    # ── Save diagnostics ──
    diag_df = pd.DataFrame({
        "feature":       feature_cols_f,
        "rmse_val":      feature_rmse_f,
        "believability": B,
        "originality":   O,
        "av_score":      B * O,
        "weight_rmse":   w_rmse,
        "weight_ava":    w_ava,
        "mu":            [models[i]["mu"] for i in range(n_features) if final_mask[i]],
    })
    diag_dir = RESULTS_DIR / "diagnostics"
    diag_dir.mkdir(parents=True, exist_ok=True)
    diag_df.to_csv(
        diag_dir / f"polymodel_{feature_set}_{split_name}_diagnostics.csv",
        index=False,
    )

    return all_variant_results


# %% [markdown]
# ## Run All Experiments

# %%
print("=" * 70)
print("  POLYMODEL: 4 Splits × 2 Feature Sets × 4 Variants")
print(f"  LNLM: {N_HERMITE} Hermite polynomials, {N_FOLDS}-fold CV, "
      f"{len(MU_GRID)} μ values")
print(f"  Target: minret_5d_pct (percentage points, NOT z-scored)")
print(f"  Derived AUC: roc_auc_score(y_binary, -y_pred) — rank-based")
print(f"  Cleanup: norm threshold = {NORM_THRESHOLD}")
print(f"  Filtering: {'naive RMSE baseline' if FILTER_BY_NAIVE else 'none'}")
print(f"  Believability: {'normalised by naive RMSE' if NORMALISE_BELIEVABILITY else 'raw ln(RMSE)^2'}")
print(f"  Results: {RESULTS_DIR}")
print("=" * 70)

all_results = []
completed   = 0
failed      = 0
total_start = time.time()

for feature_set in FEATURE_SETS:
    for split_name in ALL_SPLITS:
        try:
            variant_results = run_polymodel_experiment(split_name, feature_set)

            for variant, metrics in variant_results.items():
                test_m = metrics["test"]
                all_results.append({
                    "feature_set":      feature_set,
                    "split":            split_name,
                    "variant":          variant,
                    "test_r2":          test_m.get("r2"),
                    "test_mse":         test_m.get("mse"),
                    "test_mae":         test_m.get("mae"),
                    "test_derived_auc": test_m.get("derived_auc"),
                    "test_pred_std":    test_m.get("pred_std"),
                    "train_r2":         metrics["train"].get("r2"),
                })

            completed += 1
            print(f"\n  ✓ {completed}/8 fitting runs complete")

        except Exception as e:
            failed += 1
            print(f"\n  ✗ FAILED: {feature_set}/{split_name}: {e}")
            import traceback
            traceback.print_exc()
            continue

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/8 fitting runs, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes")
print(f"{'='*70}")

# %% [markdown]
# ## Results Summary

# %%
if all_results:
    results_df = pd.DataFrame(all_results)

    # ── R² by variant ──
    for variant in ["equal", "rmse", "ava", "ava_unc"]:
        print(f"\n{'='*70}")
        print(f"  POLYMODEL ({variant}) — Test R²")
        print(f"{'='*70}\n")
        vdf = results_df[results_df["variant"] == variant]
        if len(vdf) > 0:
            pivot = vdf.pivot_table(
                index="feature_set", columns="split", values="test_r2"
            )
            pivot["Mean"] = pivot.mean(axis=1)
            print(pivot.round(4).to_string())

    # ── MSE by variant ──
    for variant in ["equal", "rmse", "ava", "ava_unc"]:
        print(f"\n{'='*70}")
        print(f"  POLYMODEL ({variant}) — Test MSE (percent², comparable to vol baseline)")
        print(f"{'='*70}\n")
        vdf = results_df[results_df["variant"] == variant]
        if len(vdf) > 0:
            pivot = vdf.pivot_table(
                index="feature_set", columns="split", values="test_mse"
            )
            pivot["Mean"] = pivot.mean(axis=1)
            print(pivot.round(4).to_string())

    # ── Derived AUC by variant ──
    for variant in ["equal", "rmse", "ava", "ava_unc"]:
        print(f"\n{'='*70}")
        print(f"  POLYMODEL ({variant}) — Derived AUC")
        print(f"{'='*70}\n")
        vdf = results_df[results_df["variant"] == variant]
        if len(vdf) > 0:
            pivot = vdf.pivot_table(
                index="feature_set", columns="split", values="test_derived_auc"
            )
            pivot["Mean"] = pivot.mean(axis=1)
            print(pivot.round(4).to_string())

    # ── Variant comparison ──
    print(f"\n{'='*70}")
    print(f"  VARIANT COMPARISON — Mean Derived AUC across splits")
    print(f"{'='*70}\n")
    summary = results_df.groupby(
        ["feature_set", "variant"]
    )["test_derived_auc"].mean()
    print(summary.unstack("variant").round(4).to_string())

    # ── Prediction std sanity ──
    # The target has a standard deviation of roughly 1.2 percentage points, so a
    # model carrying any signal should produce a prediction std around 0.3–0.5.
    # The threshold below is calibrated for that scale, not for a unit-variance
    # z-scored target.
    print(f"\n{'='*70}")
    print(f"  CONTINUOUS SANITY: Prediction std (AVA variant, > 0.1 in percent space)")
    print(f"{'='*70}\n")
    ava_df = results_df[results_df["variant"] == "ava"]
    for _, row in ava_df.iterrows():
        flag = "⚠ NEAR CONSTANT" if row.get("test_pred_std", 1) < 0.1 else "✓"
        print(f"  {row['feature_set']:<20} {row['split']:<10}  "
              f"pred_std={row.get('test_pred_std', 0):.4f}  {flag}")

    # ── Overfitting check ──
    print(f"\n{'='*70}")
    print(f"  OVERFITTING CHECK — Train R² vs Test R² (AVA)")
    print(f"{'='*70}\n")
    for _, row in ava_df.iterrows():
        gap  = (row["train_r2"] or 0) - (row["test_r2"] or 0)
        flag = " ⚠" if gap > 0.10 else ""
        print(f"  {row['feature_set']:<20} {row['split']:<10} "
              f"Train={row['train_r2']:.4f}  Test={row['test_r2']:.4f}  "
              f"Gap={gap:+.4f}{flag}")
else:
    print("  No results to display.")

# %% [markdown]
# ## Backtest (AVA Variant)
#
# The AVA variant is used for backtesting — it is the theoretically
# strongest aggregation (believability × originality weighting) and
# the primary result for the polymodel. The other three variants are
# evaluated and saved above; their metrics appear in the summary tables.
#
# This is a continuous-only model so go_cash_when="below":
# a more negative predicted drawdown means higher crash risk, so go to cash.
#
# run_full_backtest produces all six sections automatically:
#   signal diagnostics, simple timing, timing cost sweep,
#   asymmetric risk-scaled, risk-scaled cost sweep, side-by-side.

# %%
print("\n" + "=" * 70)
print("  POLYMODEL AVA — BACKTESTS")
print("  Signal: predicted minret_5d_pct (go to cash when signal LOW)")
print("=" * 70)

for feature_set in FEATURE_SETS:
    for split_name in ALL_SPLITS:
        try:
            loaded_val  = load_predictions(
                f"polymodel_ava_{feature_set}", split_name,
                "continuous", "val", results_dir=RESULTS_DIR,
            )
            loaded_test = load_predictions(
                f"polymodel_ava_{feature_set}", split_name,
                "continuous", "test", results_dir=RESULTS_DIR,
            )

            run_full_backtest(
                val_returns  = loaded_val["predictions"]["daily_return"].values,
                val_signal   = loaded_val["predictions"]["y_pred"].values,
                test_returns = loaded_test["predictions"]["daily_return"].values,
                test_signal  = loaded_test["predictions"]["y_pred"].values,
                go_cash_when = "below",
                model_name   = f"polymodel_ava_{feature_set}",
                split_name   = split_name,
            )

        except Exception as e:
            print(f"\n  ERROR: {feature_set}/{split_name}: {e}")
            import traceback
            traceback.print_exc()

# %% [markdown]
# ## File Inventory

# %%
print(f"\n{'='*70}")
print(f"  SAVED FILES")
print(f"{'='*70}")

for subdir in ["predictions", "metrics", "diagnostics"]:
    d = RESULTS_DIR / subdir
    if d.exists():
        files = sorted(d.glob("polymodel_*"))
        print(f"\n  {subdir}/: {len(files)} files")
        for f in files[:5]:
            print(f"    {f.name}")
        if len(files) > 5:
            print(f"    ... and {len(files) - 5} more")

# %%

  POLYMODEL: 4 Splits × 2 Feature Sets × 4 Variants
  LNLM: 4 Hermite polynomials, 10-fold CV, 101 μ values
  Target: minret_5d_pct (percentage points, NOT z-scored)
  Derived AUC: roc_auc_score(y_binary, -y_pred) — rank-based
  Cleanup: norm threshold = 50
  Filtering: naive RMSE baseline
  Believability: normalised by naive RMSE
  Results: ..\..\..\Data\Results\Dense_vs_Sparse_KAN\Polymodel

════════════════════════════════════════════════════════════
  agg_full_moments / Split_A
════════════════════════════════════════════════════════════
  Features: 1699
  Train: 2116, Val: 498, Test: 503
  y_train range: [-8.94, 0.89] (percentage points), sd=1.304

  Fitting 1699 LNLMs (101 μ values, 10-fold CV)...
    Fitted 1/1699 (μ=0.98 for dlyretx_cwmean)
    Fitted 500/1699 (μ=0.59 for turnover_5d_mean_cwkurt)
    Fitted 1000/1699 (μ=0.96 for DelFINL_cwstd)
    Fitted 1500/1699 (μ=0.99 for implied_return_chg_1m_cwskew)
    Screened out: 0/1699
  Fitting complete in 32.3s
  μ distribution: me

In [3]:
# %% [markdown]
# ## Per-Factor Univariate R²
#
# The pipeline discards `models` and saves rmse_val only for filter survivors,
# so per-factor test R² can't be recovered from the saved diagnostics. Refitting
# is deterministic and reproduces the same models.
#
# Requires results_df from the main run (for the aggregate comparison).
# Runtime ~4 minutes.
# %%
from data_utils import load_theme_assignment

THEMES_DIR = SPLITS_DIR.parent / "05_themes"


def per_factor_r2(split_name, feature_set):
    """One row per feature: mu and out-of-sample test R2."""
    d = load_split(split_name, feature_set, SPLITS_DIR)
    models, y_mean = fit_all_lnlms(d["X_train"], d["minret_train"], d["feature_cols"])
    clean = cleanup_models(models, d["feature_cols"])
    y = d["minret_test"]
    preds = predict_all(d["X_test"], models, y_mean)
    sst = np.sum((y - y.mean()) ** 2)
    return pd.DataFrame({
        "feature_set": feature_set,
        "split": split_name,
        "feature": d["feature_cols"],
        "mu": [m["mu"] for m in models],
        "r2_test": [1 - np.sum((y - preds[:, p]) ** 2) / sst if clean[p] else np.nan
                    for p in range(d["n_features"])],
    })


print("=" * 110)
print("  PER-FACTOR UNIVARIATE R^2")
print("=" * 110)
pf = pd.concat([per_factor_r2(s, fs) for fs in FEATURE_SETS for s in ALL_SPLITS],
               ignore_index=True)

# ── attach themes: join on the FULL column name, per dataset ────────────────
#
# EDIT: the previous approach stripped the moment suffix (_cwmean, _cwstd,
# etc.) and joined against a taxonomy DEDUPLICATED to one row per
# base_factor. That was correct under the old taxonomy, where every moment
# of a base factor shared one subtheme, but is WRONG under the new one:
# a base factor's cwmean and cwstd variants can sit in genuinely different
# subthemes (e.g. "Analyst Coverage Breadth" vs "Analyst Coverage Breadth
# Std"). Stripping the suffix threw that distinction away and every moment
# of a factor silently inherited whichever subtheme happened to survive
# drop_duplicates.
#
# The fix is to not strip anything. load_theme_assignment()'s 'column'
# field is already the exact feature name for each dataset -- moment-
# suffixed for agg_full_moments, bare for agg_means -- so joining on it
# directly is both correct and needs no regex, no isin() check, and no
# deduplication.
taxonomies = {fs: load_theme_assignment(fs, THEMES_DIR) for fs in FEATURE_SETS}

pf_parts = []
for fs in FEATURE_SETS:
    tax = taxonomies[fs][["column", "theme_name", "subtheme_name"]]
    part = pf[pf.feature_set == fs].merge(
        tax, left_on="feature", right_on="column", how="left"
    )
    pf_parts.append(part)
pf = pd.concat(pf_parts, ignore_index=True)

n_unmatched = pf["theme_name"].isna().sum()
if n_unmatched:
    print(f"  NOTE: {n_unmatched} feature-rows have no taxonomy match "
          f"(expected only for features deliberately excluded from the "
          f"taxonomy, e.g. dropped binaries -- verify this count matches "
          f"that expectation before trusting the theme/subtheme tables below)")

# ── one row per feature, R2 per split in columns ──
fac = pf.pivot_table(index=["feature_set", "feature"], columns="split",
                     values="r2_test").reset_index()
fac["mean"] = fac[ALL_SPLITS].mean(axis=1)
fac["n_pos"] = (fac[ALL_SPLITS] > 0).sum(axis=1)
fac = fac.merge(
    pf.groupby(["feature_set", "feature"])[["theme_name", "subtheme_name"]].first(),
    on=["feature_set", "feature"], how="left")

F = {c: "{:+.4f}".format for c in ALL_SPLITS + ["mean"]}
COLS = ["feature", "theme_name", "subtheme_name"] + ALL_SPLITS + ["mean", "n_pos"]
GCOLS = lambda g: [g, "n"] + ALL_SPLITS + ["mean"]


def group_table(df, key, label, fs):
    """
    Mean per-factor R2 within each group, per split and averaged.
    Takes the WIDE frame (one row per feature, one column per split).
    groupby drops NaN keys, so unthemed factors are excluded by design.
    """
    g = df.groupby(key).agg(n=(ALL_SPLITS[0], "size"),
                            **{s: (s, "mean") for s in ALL_SPLITS}).reset_index()
    g["mean"] = g[ALL_SPLITS].mean(axis=1)
    g = g.sort_values("mean", ascending=False)
    print(f"\n  {label} — {fs}")
    print("  " + "-" * 106)
    print(g[GCOLS(key)].to_string(index=False, formatters=F))


for fs in FEATURE_SETS:
    a = fac[fac.feature_set == fs]
    A = results_df[(results_df.variant == "ava")
                   & (results_df.feature_set == fs)]["test_r2"].mean()

    print(f"\n{'=' * 110}")
    print(f"  {fs}   {len(a)} factors   unmatched to theme: {a.theme_name.isna().sum()}")
    print("=" * 110)
    print(f"  mean per-factor R2    {a['mean'].mean():+.4f}")
    print(f"  median                {a['mean'].median():+.4f}")
    print(f"  best single factor    {a['mean'].max():+.4f}")
    print(f"  R2 > 0                {(a['mean'] > 0).sum()} of {len(a)}")
    print(f"  positive in 4/4       {(a.n_pos == 4).sum()}")
    print(f"  AGGREGATE (AVA)       {A:+.4f}")

    print("\n  MEAN PER-FACTOR R2 BY SPLIT")
    print("  " + "  ".join(f"{s} {a[s].mean():+.4f}" for s in ALL_SPLITS))

    print("\n  QUANTILES of mean R2")
    print(a["mean"].quantile([.01, .05, .25, .50, .75, .95, .99])
           .to_frame("r2").to_string(formatters={"r2": "{:+.4f}".format}))

    group_table(a, "theme_name", "BY THEME", fs)
    group_table(a, "subtheme_name", "BY SUBTHEME", fs)

    print(f"\n  TOP 50 FACTORS — {fs}")
    print("  " + "-" * 106)
    print(a.nlargest(50, "mean")[COLS].to_string(index=False, formatters=F))

    print(f"\n  BOTTOM 50 FACTORS — {fs}")
    print("  " + "-" * 106)
    print(a.nsmallest(50, "mean")[COLS].to_string(index=False, formatters=F))

fac.to_csv(RESULTS_DIR / "diagnostics" / "per_factor_r2.csv", index=False)
print(f"\n  saved -> {RESULTS_DIR / 'diagnostics' / 'per_factor_r2.csv'}")


# ── PER-FACTOR MSE — derived from the saved R², no refit ─────────────────────
#
# MSE = (1 - R2) * Var(y_test), exactly, since R2 = 1 - SSE/SST and
# SST = n * Var(y_test). Recovering var_Y per split is the only thing needed.
#
# MSE is in percent^2 and is not rescaled by test-period variance, so it is
# comparable across splits in a way R2 is not. Split D's target variance is
# roughly a sixth of Split B's, which is what distorts the R2 average.
fac = pd.read_csv(RESULTS_DIR / "diagnostics" / "per_factor_r2.csv")

# target variance per split, from the split files
varY = {s: load_split(s, FEATURE_SETS[0], SPLITS_DIR)["minret_test"].var()
        for s in ALL_SPLITS}

print("=" * 110)
print("  PER-FACTOR MSE  (percent^2, derived from saved R^2)")
print("=" * 110)
print("  test-period target variance:  "
      + "   ".join(f"{s} {varY[s]:.4f}" for s in ALL_SPLITS))

MSE_COLS = [f"mse_{s}" for s in ALL_SPLITS]
for s in ALL_SPLITS:
    fac[f"mse_{s}"] = (1 - fac[s]) * varY[s]
fac["mse_mean"] = fac[MSE_COLS].mean(axis=1)

# naive benchmark: predict the test mean, i.e. MSE = var_Y, R2 = 0
naive = np.mean([varY[s] for s in ALL_SPLITS])
fac["beats_naive"] = (fac[MSE_COLS] < [varY[s] for s in ALL_SPLITS]).sum(axis=1)

FM = {c: "{:.4f}".format for c in MSE_COLS + ["mse_mean"]}
COLS_M = (["feature", "theme_name", "subtheme_name"]
          + MSE_COLS + ["mse_mean", "beats_naive"])

for fs in FEATURE_SETS:
    a = fac[fac.feature_set == fs]
    A_mse = results_df[(results_df.variant == "ava")
                       & (results_df.feature_set == fs)]["test_mse"].mean()

    print(f"\n{'=' * 110}")
    print(f"  {fs}   {len(a)} factors")
    print("=" * 110)
    print(f"  median per-factor MSE   {a.mse_mean.median():.4f}")
    print(f"  best single factor      {a.mse_mean.min():.4f}")
    print(f"  naive (predict mean)    {naive:.4f}")
    print(f"  AGGREGATE (AVA)         {A_mse:.4f}")
    print(f"  factors beating naive in all 4 splits   "
          f"{(a.beats_naive == 4).sum()} of {len(a)}")

    print("\n  MEDIAN PER-FACTOR MSE BY SPLIT")
    print("  " + "  ".join(f"{s} {a[f'mse_{s}'].median():.4f}" for s in ALL_SPLITS))
    print("  naive by split          "
          + "  ".join(f"{s} {varY[s]:.4f}" for s in ALL_SPLITS))

    print("\n  QUANTILES of mean MSE")
    print(a.mse_mean.quantile([.01, .05, .25, .50, .75, .95, .99])
           .to_frame("mse").to_string(formatters={"mse": "{:.4f}".format}))

    # theme table on medians — the mean is destroyed by extrapolation blowups
    g = (a.groupby("theme_name")
          .agg(n=("mse_mean", "size"),
               **{f"mse_{s}": (f"mse_{s}", "median") for s in ALL_SPLITS},
               mse_mean=("mse_mean", "median"))
          .sort_values("mse_mean").reset_index())
    print(f"\n  BY THEME — median MSE — {fs}")
    print("  " + "-" * 106)
    print(g.to_string(index=False, formatters=FM))

    print(f"\n  TOP 30 FACTORS BY MEAN MSE — {fs}")
    print("  " + "-" * 106)
    print(a.nsmallest(30, "mse_mean")[COLS_M].to_string(index=False, formatters=FM))

fac.to_csv(RESULTS_DIR / "diagnostics" / "per_factor_r2_mse.csv", index=False)
print(f"\n  saved -> {RESULTS_DIR / 'diagnostics' / 'per_factor_r2_mse.csv'}")


# %% [markdown]
# ## Subtheme Composite Test (Spearman–Brown)
#
# Builds one sign-aligned composite per subtheme, fits a single LNLM to it, and
# compares against its members individually. Tests whether grouping amplifies
# signal as the reliability formula predicts.
#
# Signs come from the first eigenvector of the train correlation matrix, so the
# alignment never touches the target. Composites are standardised on train stats
# because the Hermite basis assumes a roughly standard-normal input.
#
# Requires pf from the per-factor cell (now correctly labelled per exact
# column name, not per stripped base_factor -- see the taxonomy join fix
# above). A subtheme composite mixing e.g. a factor's cwmean AND its cwstd
# sibling would previously have been possible only by accident of the old
# mislabeling; now every member genuinely belongs to the named subtheme.
#
# Runtime ~1 minute.
# %%
MIN_MEMBERS = 3


def composite_test(split_name, feature_set):
    d = load_split(split_name, feature_set, SPLITS_DIR)
    X_tr, X_te = d["X_train"], d["X_test"]
    y_tr, y_te = d["minret_train"], d["minret_test"]
    col_ix = {c: i for i, c in enumerate(d["feature_cols"])}
    y_mean = y_tr.mean()
    y_c = y_tr - y_mean
    folds = stratified_kfold(y_c, k=N_FOLDS, seed=42)
    sst = np.sum((y_te - y_te.mean()) ** 2)

    sub = (pf[(pf.feature_set == feature_set) & (pf.split == split_name)]
           .dropna(subset=["subtheme_name"]))

    rows = []
    for name, grp in sub.groupby("subtheme_name"):
        ix = [col_ix[f] for f in grp.feature]
        if len(ix) < MIN_MEMBERS:
            continue

        keep = X_tr[:, ix].std(axis=0) > 1e-10
        if keep.sum() < MIN_MEMBERS:
            continue
        ix = [i for i, k in zip(ix, keep) if k]
        grp = grp.iloc[np.where(keep)[0]]
        m = len(ix)

        A_tr = X_tr[:, ix].astype(np.float64)
        A_te = X_te[:, ix].astype(np.float64)

        # sign-align on the train correlation matrix (target-free)
        C = np.nan_to_num(np.corrcoef(A_tr, rowvar=False), nan=0.0)
        w, V = np.linalg.eigh(C)
        v = V[:, -1]
        s = np.sign(v)
        s[s == 0] = 1.0
        lam = w[::-1]
        eig_ratio = lam[0] / lam[1] if lam[1] > 1e-10 else np.inf
        eig_share = lam[0] / lam.sum()

        # equal-weight composite, standardised on train stats
        c_tr, c_te = (A_tr * s).mean(axis=1), (A_te * s).mean(axis=1)
        mu, sd = c_tr.mean(), c_tr.std()
        if sd < 1e-10:
            continue
        c_tr, c_te = (c_tr - mu) / sd, (c_te - mu) / sd

        # eigenvector-weighted composite
        d_tr, d_te = A_tr @ v, A_te @ v
        mu_w, sd_w = d_tr.mean(), d_tr.std()
        if sd_w > 1e-10:
            d_tr, d_te = (d_tr - mu_w) / sd_w, (d_te - mu_w) / sd_w
            ms_w, bl_w, bn_w = fit_single_factor(d_tr, y_c, folds, k=N_FOLDS)
            pred_w = lnlm_predict(d_te, {"mu": ms_w, "beta_lin": bl_w,
                                         "beta_nonlin": bn_w}, y_mean)
            r2_comp_w = 1 - np.sum((y_te - pred_w) ** 2) / sst
            corr_comp_w = abs(np.corrcoef(d_te, y_te)[0, 1])
        else:
            r2_comp_w, corr_comp_w = np.nan, np.nan

        # average pairwise correlation after alignment, and m_eff
        up = np.triu_indices(m, 1)
        rho = float((C * np.outer(s, s))[up].mean())
        denom = 1 + (m - 1) * rho
        m_eff = m / denom if (rho > 0 and denom > 1e-6) else np.nan

        cm = np.array([np.corrcoef(A_te[:, j] * s[j], y_te)[0, 1] for j in range(m)])
        cm = np.nan_to_num(cm, nan=0.0)
        cm_signed, cm_abs = abs(cm.mean()), np.abs(cm).mean()
        sign_coh = cm_signed / cm_abs if cm_abs > 1e-10 else np.nan

        ms, bl, bn = fit_single_factor(c_tr, y_c, folds, k=N_FOLDS)
        pred = lnlm_predict(c_te, {"mu": ms, "beta_lin": bl,
                                   "beta_nonlin": bn}, y_mean)

        rows.append({
            "feature_set": feature_set, "split": split_name,
            "subtheme_name": name, "n": m, "rho_bar": rho, "m_eff": m_eff,
            "eig_ratio": eig_ratio, "eig_share": eig_share,
            "corr_mem_signed": cm_signed, "corr_mem_abs": cm_abs,
            "sign_coherence": sign_coh,
            "corr_comp": abs(np.corrcoef(c_te, y_te)[0, 1]),
            "corr_comp_w": corr_comp_w,
            "r2_comp": 1 - np.sum((y_te - pred) ** 2) / sst,
            "r2_comp_w": r2_comp_w,
            "r2_mem_med": grp.r2_test.median(),
            "r2_mem_best": grp.r2_test.max(),
        })
    return pd.DataFrame(rows)


print("=" * 110)
print("  SUBTHEME COMPOSITE TEST (SPEARMAN-BROWN)")
print("=" * 110)

comp = pd.concat([composite_test(s, fs) for fs in FEATURE_SETS for s in ALL_SPLITS],
                 ignore_index=True)
comp["amp_obs"] = comp.corr_comp / comp.corr_mem_signed.replace(0, np.nan)
comp["amp_obs_abs"] = comp.corr_comp / comp.corr_mem_abs.replace(0, np.nan)
comp["amp_pred"] = np.sqrt(comp.m_eff)
comp.loc[comp.sign_coherence < 0.3, "amp_obs"] = np.nan   # unreliable denominator

F = {c: "{:+.4f}".format for c in
     ["rho_bar", "corr_mem", "corr_comp", "r2_comp", "r2_mem_med", "r2_mem_best"]}
F.update({c: "{:.2f}".format for c in ["m_eff", "amp_obs", "amp_pred"]})

for fs in FEATURE_SETS:
    c = comp[comp.feature_set == fs]
    print(f"\n{'=' * 110}")
    print(f"  {fs}   {c.subtheme_name.nunique()} subthemes with >= {MIN_MEMBERS} members")
    print("=" * 110)
    print(f"  median rho_bar (within-subtheme correlation)   {c.rho_bar.median():+.3f}")
    print(f"  median m_eff (effective independent readings)  {c.m_eff.median():.2f}")
    print(f"  predicted amplification  sqrt(m_eff)           {c.amp_pred.median():.2f}x")
    print(f"  OBSERVED amplification   corr_comp/corr_mem    {c.amp_obs.median():.2f}x")

    print(f"\n  composite R2 beats MEDIAN member   "
          f"{(c.r2_comp > c.r2_mem_med).mean():.0%} of subtheme-splits")
    print(f"  composite R2 beats BEST member     "
          f"{(c.r2_comp > c.r2_mem_best).mean():.0%} of subtheme-splits")
    print(f"  composite R2 > 0                   "
          f"{(c.r2_comp > 0).mean():.0%} of subtheme-splits")

    g = (c.groupby("subtheme_name")
          .agg(n=("n", "first"), rho_bar=("rho_bar", "mean"),
               m_eff=("m_eff", "mean"), amp_pred=("amp_pred", "mean"),
               amp_obs=("amp_obs", "mean"), r2_comp=("r2_comp", "mean"),
               r2_mem_med=("r2_mem_med", "median"),
               r2_mem_best=("r2_mem_best", "max"))
          .sort_values("r2_comp", ascending=False).reset_index())
    print(f"\n  BY SUBTHEME — mean across splits, sorted by composite R2")
    print("  " + "-" * 106)
    print(g.to_string(index=False, formatters=F))

comp.to_csv(RESULTS_DIR / "diagnostics" / "subtheme_composite.csv", index=False)
print(f"\n  saved -> {RESULTS_DIR / 'diagnostics' / 'subtheme_composite.csv'}")

  PER-FACTOR UNIVARIATE R^2
    Fitted 1/1699 (μ=0.98 for dlyretx_cwmean)
    Fitted 500/1699 (μ=0.59 for turnover_5d_mean_cwkurt)
    Fitted 1000/1699 (μ=0.96 for DelFINL_cwstd)
    Fitted 1500/1699 (μ=0.99 for implied_return_chg_1m_cwskew)
    Screened out: 0/1699
    Cleanup: 0 invalid, 0 explosive, 0 inf/NaN → 1699/1699 kept
    Fitted 1/1699 (μ=0.97 for dlyretx_cwmean)
    Fitted 500/1699 (μ=0.00 for turnover_5d_mean_cwkurt)
    Fitted 1000/1699 (μ=1.00 for DelFINL_cwstd)
    Fitted 1500/1699 (μ=0.97 for implied_return_chg_1m_cwskew)
    Screened out: 0/1699
    Cleanup: 0 invalid, 0 explosive, 0 inf/NaN → 1699/1699 kept
    Fitted 1/1699 (μ=0.97 for dlyretx_cwmean)
    Fitted 500/1699 (μ=0.00 for turnover_5d_mean_cwkurt)
    Fitted 1000/1699 (μ=1.00 for DelFINL_cwstd)
    Fitted 1500/1699 (μ=0.97 for implied_return_chg_1m_cwskew)
    Screened out: 0/1699
    Cleanup: 0 invalid, 0 explosive, 0 inf/NaN → 1699/1699 kept
    Fitted 1/1699 (μ=0.97 for dlyretx_cwmean)
    Fitted 500/16